# Official checkpoint strict load

This notebook constructs the published CLIP-L/14 vision architecture offline and strictly loads a complete DFD-HR checkpoint. It does not read a dataset or run training.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch
import yaml


def required_path(name):
    value = os.environ.get(name)
    assert value, f"Missing required environment variable: {name}"
    path = Path(value).expanduser().resolve()
    assert path.exists(), f"{name} does not exist"
    return path


repo_root = required_path("DFDHR_REPO_ROOT")
checkpoint_path = required_path("DFDHR_OFFICIAL_CHECKPOINT")
assert repo_root == Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip()).resolve()
sys.path.insert(0, str(repo_root / "training"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("Python:", sys.executable)
print("Git commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip())


In [ ]:
from detectors import DETECTOR
from evaluation_utils import load_checkpoint_strict, sha256_file

config_path = repo_root / "training/config/detector/dfd_hr.yaml"
with config_path.open(encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
config["backbone_pretrained"] = False

print("Checkpoint file:", checkpoint_path.name)
print("Checkpoint size (bytes):", checkpoint_path.stat().st_size)
print("Checkpoint SHA-256:", sha256_file(checkpoint_path))
print("Detector config SHA-256:", sha256_file(config_path))


In [ ]:
model = DETECTOR[config["model_name"]](config)
load_result = load_checkpoint_strict(model, checkpoint_path)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("Strict load: OK")
print("Checkpoint tensors:", load_result["tensor_count"])
print("Model parameters:", parameter_count)
assert load_result["tensor_count"] == len(model.state_dict())
